# DATA 255 Lab 2: Same-Resolution Residual Attention Model
**Student Name:** [Your Name]  
**Date:** Spring 2026

## Notebook Role
This notebook is a functional prototype for the Lab 2 project. It mirrors the project structure you eventually want in `src/`, `train.py`, and `export_onnx.py`, but keeps the full workflow in one place so you can iterate quickly.

## Covered in This Notebook
- one-time split-folder canonicalization
- paired LR/HR validation and loading
- same-resolution residual-attention model
- L1 training + validation PSNR
- best/last checkpoint saving
- fixed-shape ONNX export + optional ONNX Runtime parity check


In [ ]:
from pathlib import Path
import json
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

from src.data import (
    PairedImageDataset,
    collect_paired_samples,
    load_rgb_tensor,
    rename_legacy_split_dirs,
    require_canonical_split_dirs,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Dataset Preparation
The project contract uses canonical split names:

- `training_data/LR_train`
- `training_data/HR_train`
- `training_data/LR_val`
- `training_data/HR_val`

If the original folders still exist (`LR`, `HR`, `LR 2`, `HR 2`), the helper below renames them once.


In [ ]:
DATA_ROOT = Path("training_data")
OUTPUT_DIR = Path("runs/notebook_baseline")
AUTO_RENAME_LEGACY_SPLITS = True

legacy_dirs = [DATA_ROOT / name for name in ("LR", "HR", "LR 2", "HR 2")]
if AUTO_RENAME_LEGACY_SPLITS and any(path.exists() for path in legacy_dirs):
    rename_legacy_split_dirs(DATA_ROOT)
    print("Renamed legacy split folders to canonical names.")

split_dirs = require_canonical_split_dirs(DATA_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_path in split_dirs.items():
    print(f"{split_name}: {split_path}")


In [ ]:
def verify_split(lr_dir: Path, hr_dir: Path, split_name: str) -> None:
    pairs = collect_paired_samples(lr_dir, hr_dir)
    for sample in pairs:
        load_rgb_tensor(sample.lr_path)
        load_rgb_tensor(sample.hr_path)
    print(f"[{split_name}] pairs={len(pairs)} | all images load as RGB 256x256")

verify_split(split_dirs["LR_train"], split_dirs["HR_train"], "train")
verify_split(split_dirs["LR_val"], split_dirs["HR_val"], "val")


## 2. Paired Loading
The dataset class lives in `src.data` and does the required synchronized train-only augmentation: random horizontal flip, random vertical flip, and random `k * 90` rotation applied identically to LR and HR.


In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 0
TINY_OVERFIT_SAMPLES = None  # set to 8 or 16 for debugging
SEED = 255

train_dataset = PairedImageDataset(
    split_dirs["LR_train"],
    split_dirs["HR_train"],
    train=True,
    subset_size=TINY_OVERFIT_SAMPLES,
    seed=SEED,
)
val_dataset = PairedImageDataset(
    split_dirs["LR_val"],
    split_dirs["HR_val"],
    train=False,
    subset_size=TINY_OVERFIT_SAMPLES,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

lr_batch, hr_batch, names = next(iter(train_loader))
print("train dataset size:", len(train_dataset))
print("val dataset size:", len(val_dataset))
print("lr batch shape:", tuple(lr_batch.shape), lr_batch.dtype, float(lr_batch.min()), float(lr_batch.max()))
print("hr batch shape:", tuple(hr_batch.shape), hr_batch.dtype, float(hr_batch.min()), float(hr_batch.max()))
print("sample basenames:", list(names[:3]))


## 3. Model Definition
This implementation stays export-safe by using only convolution/depthwise convolution, adaptive average pooling, PReLU, HardSigmoid, add, and multiply. There is no clamp inside the model graph.


In [ ]:
FORBIDDEN_TYPES = (
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.BatchNorm3d,
    nn.ReLU,
    nn.Sigmoid,
    nn.Softmax,
    nn.Upsample,
)

def conv3x3(in_channels: int, out_channels: int) -> nn.Conv2d:
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

class ResidualBlock(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.expand = nn.Conv2d(48, 96, kernel_size=1)
        self.depthwise = nn.Conv2d(96, 96, kernel_size=kernel_size, padding=padding, groups=96)
        self.act = nn.PReLU(num_parameters=96)
        self.project = nn.Conv2d(96, 48, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.expand(x)
        x = self.depthwise(x)
        x = self.act(x)
        x = self.project(x)
        return residual + x

class StageAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.reduce = nn.Conv2d(48, 12, kernel_size=1)
        self.act = nn.PReLU(num_parameters=12)
        self.expand = nn.Conv2d(12, 48, kernel_size=1)
        self.gate = nn.Hardsigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = self.reduce(scale)
        scale = self.act(scale)
        scale = self.expand(scale)
        scale = self.gate(scale)
        return x * scale

class ResidualGroup(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(
            ResidualBlock(kernel_size=3),
            ResidualBlock(kernel_size=5),
            ResidualBlock(kernel_size=7),
        )
        self.attention = StageAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.blocks(x)
        x = self.attention(x)
        return residual + x

class MultiScaleResidualAttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            conv3x3(3, 48),
            nn.PReLU(num_parameters=48),
        )
        self.body = nn.Sequential(
            ResidualGroup(),
            ResidualGroup(),
            ResidualGroup(),
        )
        self.tail = nn.Sequential(
            conv3x3(48, 24),
            nn.PReLU(num_parameters=24),
            conv3x3(24, 3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        image_residual = x
        stem_features = self.stem(x)
        body_features = self.body(stem_features)
        fused = body_features + stem_features
        delta = self.tail(fused)
        return image_residual + delta

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def assert_no_forbidden_modules(model: nn.Module) -> None:
    for module in model.modules():
        if isinstance(module, FORBIDDEN_TYPES):
            raise TypeError(f"Forbidden module found: {module.__class__.__name__}")

model = MultiScaleResidualAttentionNet().to(device)
assert_no_forbidden_modules(model)

with torch.no_grad():
    smoke_input = torch.randn(1, 3, 256, 256, device=device)
    smoke_output = model(smoke_input)

print("parameter count:", count_parameters(model))
print("forward output shape:", tuple(smoke_output.shape))


## 4. Metrics and Training Helpers
Validation PSNR is computed on clamped `[0, 1]` predictions. The exported model remains unclamped.


In [ ]:
TRAIN_CFG = {
    "epochs": 80,
    "batch_size": 8,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "min_lr_ratio": 0.05,
    "num_workers": NUM_WORKERS,
    "seed": SEED,
}
print(TRAIN_CFG)

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_psnr(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(eps)
    return -10.0 * torch.log10(mse)

def lr_for_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int, min_lr_ratio: float) -> float:
    if total_epochs <= 0:
        raise ValueError("total_epochs must be positive")
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    decay_epochs = max(1, total_epochs - warmup_epochs)
    progress = float(epoch - warmup_epochs) / float(decay_epochs - 1 if decay_epochs > 1 else 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_lr_ratio
    return min_lr + (base_lr - min_lr) * cosine

def set_optimizer_lr(optimizer: torch.optim.Optimizer, lr: float) -> None:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    total_items = 0

    for lr_img, hr_img, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(lr_img)
        loss = F.l1_loss(pred, hr_img)
        loss.backward()
        optimizer.step()

        batch_size = lr_img.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    return total_loss / max(1, total_items)

def validate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_items = 0

    with torch.no_grad():
        for lr_img, hr_img, _ in loader:
            lr_img = lr_img.to(device, non_blocking=True)
            hr_img = hr_img.to(device, non_blocking=True)

            pred = model(lr_img)
            loss = F.l1_loss(pred, hr_img)
            psnr = compute_psnr(pred, hr_img)

            if not torch.isfinite(psnr).all():
                raise FloatingPointError("Validation PSNR produced non-finite values")

            batch_size = lr_img.size(0)
            total_loss += loss.item() * batch_size
            total_psnr += psnr.sum().item()
            total_items += batch_size

    return {
        "val_loss": total_loss / max(1, total_items),
        "val_psnr": total_psnr / max(1, total_items),
    }

def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, metrics: dict) -> None:
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "train_cfg": TRAIN_CFG,
    }
    torch.save(payload, path)

def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    output_dir: Path,
    epochs: int = TRAIN_CFG["epochs"],
    lr: float = TRAIN_CFG["lr"],
    weight_decay: float = TRAIN_CFG["weight_decay"],
    warmup_epochs: int = TRAIN_CFG["warmup_epochs"],
    min_lr_ratio: float = TRAIN_CFG["min_lr_ratio"],
    seed: int = TRAIN_CFG["seed"],
):
    set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    metrics_path.write_text("")

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_psnr = float("-inf")
    history = []

    for epoch in range(epochs):
        epoch_lr = lr_for_epoch(epoch, epochs, lr, warmup_epochs, min_lr_ratio)
        set_optimizer_lr(optimizer, epoch_lr)
        start_time = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = validate(model, val_loader, device)
        elapsed = time.time() - start_time

        row = {
            "epoch": epoch + 1,
            "lr": epoch_lr,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "val_psnr": val_metrics["val_psnr"],
            "seconds": elapsed,
        }
        history.append(row)
        with metrics_path.open("a") as handle:
            handle.write(json.dumps(row) + "\n")

        save_checkpoint(output_dir / "last.pt", model, optimizer, epoch + 1, row)
        if row["val_psnr"] > best_psnr:
            best_psnr = row["val_psnr"]
            save_checkpoint(output_dir / "best.pt", model, optimizer, epoch + 1, row)

        print(
            f"epoch {epoch + 1:03d}/{epochs:03d} | "
            f"lr={epoch_lr:.2e} | "
            f"train_l1={train_loss:.6f} | "
            f"val_l1={row['val_loss']:.6f} | "
            f"val_psnr={row['val_psnr']:.3f} dB | "
            f"time={elapsed:.1f}s"
        )

    return history


In [ ]:
RUN_TINY_SMOKE_TEST = True

if RUN_TINY_SMOKE_TEST:
    smoke_train = PairedImageDataset(
        split_dirs["LR_train"],
        split_dirs["HR_train"],
        train=True,
        subset_size=8,
        seed=SEED,
    )
    smoke_val = PairedImageDataset(
        split_dirs["LR_val"],
        split_dirs["HR_val"],
        train=False,
        subset_size=8,
        seed=SEED,
    )
    smoke_train_loader = DataLoader(smoke_train, batch_size=4, shuffle=True, num_workers=0)
    smoke_val_loader = DataLoader(smoke_val, batch_size=4, shuffle=False, num_workers=0)
    smoke_model = MultiScaleResidualAttentionNet().to(device)
    smoke_optimizer = AdamW(smoke_model.parameters(), lr=TRAIN_CFG["lr"], weight_decay=TRAIN_CFG["weight_decay"])
    smoke_train_loss = train_one_epoch(smoke_model, smoke_train_loader, smoke_optimizer, device)
    smoke_val_metrics = validate(smoke_model, smoke_val_loader, device)
    print("smoke train loss:", smoke_train_loss)
    print("smoke val metrics:", smoke_val_metrics)


## 5. ONNX Export
Export uses a fixed input shape `[1, 3, 256, 256]`, opset 13, named tensors `input` and `output`, and no dynamic axes.


In [ ]:
def load_checkpoint_state(model: nn.Module, checkpoint_path: Path, map_location: str | torch.device = "cpu"):
    checkpoint = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    return checkpoint

def export_checkpoint_to_onnx(checkpoint_path, onnx_out, sample_image=None, verify=False):
    checkpoint_path = Path(checkpoint_path)
    onnx_out = Path(onnx_out)
    onnx_out.parent.mkdir(parents=True, exist_ok=True)

    model = MultiScaleResidualAttentionNet().to(device)
    load_checkpoint_state(model, checkpoint_path, map_location=device)
    model.eval()

    dummy_input = torch.randn(1, 3, 256, 256, device=device)
    export_kwargs = dict(
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
    )

    try:
        torch.onnx.export(model, dummy_input, onnx_out, dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(model, dummy_input, onnx_out, **export_kwargs)

    print(f"Exported ONNX model to {onnx_out}")

    if onnx is not None:
        onnx_model = onnx.load(str(onnx_out))
        onnx.checker.check_model(onnx_model)
        print("ONNX checker passed")
    else:
        print("onnx package not installed; skipped checker")

    if verify:
        if ort is None:
            raise ImportError("onnxruntime is required for parity verification")
        if sample_image is None:
            raise ValueError("sample_image is required when verify=True")

        sample_tensor = load_rgb_tensor(sample_image).unsqueeze(0).to(device)
        with torch.no_grad():
            torch_output = model(sample_tensor).cpu().numpy()

        session = ort.InferenceSession(str(onnx_out), providers=["CPUExecutionProvider"])
        ort_output = session.run(["output"], {"input": sample_tensor.cpu().numpy()})[0]
        diff = abs(torch_output - ort_output)

        stats = {
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
        }
        print(stats)
        return stats

    return None


## 6. Example Next Steps
Use the cells above in this order:

1. Run the dataset cells to verify canonical folders and paired image integrity.
2. Run the model smoke test and confirm the parameter count/output shape.
3. Run the tiny smoke test first.
4. When that looks stable, call `fit(...)` with the full train/val loaders.
5. Export `best.pt` with `export_checkpoint_to_onnx(...)` and optionally verify parity on one fixed validation image.

Example training call:

```python
history = fit(model, train_loader, val_loader, OUTPUT_DIR)
```

Example export call:

```python
sample_path = next(split_dirs["LR_val"].glob("*.png"))
export_checkpoint_to_onnx(OUTPUT_DIR / "best.pt", OUTPUT_DIR / "best.onnx", sample_image=sample_path, verify=True)
```
